# Train orientation_strong seeds 0–11 + probe (Kaggle)
## One-time setup
## Run order

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo

Set `PROBE_REPO_URL` to your public Git URL or `PROBE_REPO_DIR` to a local checkout.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = os.environ.get("PROBE_REPO_DIR", "/kaggle/working/probe-capacity-invariance")
if not (Path(REPO_DIR) / "pyproject.toml").is_file():
    repo_url = os.environ.get("PROBE_REPO_URL")
    if not repo_url:
        raise RuntimeError("Set PROBE_REPO_URL or PROBE_REPO_DIR before running this notebook.")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)


## 3. Install dependencies

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")

## 4. Download Shapes3D + build the image cache

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 5. Restore checkpoints from a previous session (resume)

In [ ]:
import re, shutil
from pathlib import Path

REPO  = Path("/kaggle/working/probe-capacity-invariance")
ENC   = REPO / "results" / "encoders"
INPUT = Path("/kaggle/input")
_RID  = re.compile(r"orientation_strong_seed\d+")

def _target(name):
    n = name.lower()
    if "ckpt" in n:     return "last_ckpt.pt"
    if "backbone" in n: return "backbone.pt"
    return None

found = {}
for p in sorted(INPUT.rglob("*.pt")) if INPUT.exists() else []:
    m, tgt = _RID.search(p.as_posix()), _target(p.name)
    if m and tgt:
        found.setdefault((m.group(0), tgt), p)

if not found:
    print("No orientation_strong_seed*.pt under /kaggle/input -- fresh start "
          "(Add Input -> a prior version's output to resume).")
for (rid, tgt), src in sorted(found.items()):
    dst = ENC / rid / tgt
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dst)
    print(f"restored {rid}/{tgt:12s} <- {src.name}")

## 6. Integrity check — keep good checkpoints, purge corrupt ones

In [ ]:
import zipfile
from pathlib import Path

ENC = Path("/kaggle/working/probe-capacity-invariance/results/encoders")

def _state(p):
    if not p.exists():
        return "missing"
    try:
        return None if zipfile.ZipFile(p).testzip() is None else "corrupt"
    except Exception as e:
        return f"not-a-zip ({e})"

done, resume, fresh = [], [], []
for d in sorted(ENC.glob("orientation_strong_seed*")):
    bb, ck = d / "backbone.pt", d / "last_ckpt.pt"
    bstat = _state(bb)
    if bstat is None:
        done.append(d.name); print(f"{d.name:28s} backbone OK -> skip"); continue
    if bstat != "missing":
        bb.unlink(missing_ok=True); ck.unlink(missing_ok=True)
        fresh.append(d.name); print(f"{d.name:28s} backbone {bstat} -> PURGED (retrain)"); continue
    cstat = _state(ck)
    if cstat is None:
        resume.append(d.name); print(f"{d.name:28s} resume from last_ckpt.pt")
    else:
        ck.unlink(missing_ok=True)
        fresh.append(d.name); print(f"{d.name:28s} last_ckpt {cstat} -> from scratch")
print(f"\ndone={len(done)}  resume={len(resume)}  fresh={len(fresh)}")

## 7. Train — orientation_strong seeds 0–11 (resume-aware)

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
ENC = REPO / "results" / "encoders"
LOGS = REPO / "results" / "sweep_logs"; LOGS.mkdir(parents=True, exist_ok=True)

CONFIG = "configs/run/orientation_strong.yaml"
SEEDS = list(range(12))
N_GPUS = max(1, torch.cuda.device_count())

def run_id(seed): return f"orientation_strong_seed{seed}"
def is_done(seed): return (ENC / run_id(seed) / "backbone.pt").exists()

def _launch(seed, gpu):
    rid = run_id(seed)
    env = dict(os.environ,
               CUDA_VISIBLE_DEVICES=str(gpu),
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    log = open(LOGS / f"{rid}.log", "a")
    cmd = [sys.executable, "-m", "src.encoders.train_simclr", "--config", CONFIG,
           "--set", f"run.seed={seed}", "run.num_workers=2", "run.device=cuda"]
    p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT)
    return {"proc": p, "log": log, "rid": rid}

def run(n_gpus=N_GPUS, poll=15):
    subprocess.run(["pkill", "-9", "-f", "src.encoders.train_simclr"])
    time.sleep(2)
    queue = [s for s in SEEDS if not is_done(s)]
    print(f"{len(SEEDS)} seeds | {len(SEEDS)-len(queue)} done | {len(queue)} to run on {n_gpus} GPU(s)")
    slots = [None] * n_gpus
    try:
        while queue or any(slots):
            for g in range(n_gpus):
                if slots[g] is None and queue:
                    seed = queue.pop(0)
                    if is_done(seed):
                        continue
                    slots[g] = _launch(seed, g)
                    print(f"[GPU{g}] start  {slots[g]['rid']}", flush=True)
            time.sleep(poll)
            for g in range(n_gpus):
                s = slots[g]
                if s and s["proc"].poll() is not None:
                    s["log"].close()
                    seed = int(s["rid"].split("seed")[1])
                    ok = is_done(seed)
                    print(f"[GPU{g}] {'DONE ' if ok else 'EXIT(rc=%s) ' % s['proc'].returncode}{s['rid']}", flush=True)
                    slots[g] = None
    except KeyboardInterrupt:
        for s in slots:
            if s: s["proc"].terminate()
        print("interrupted — checkpoints are saved; re-run this cell to resume")
        return
    print("done — all 12 seeds have a backbone.pt")

run()

## 8. Monitor

In [ ]:
!nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv

## 9. Progress + final diagnostics

In [ ]:
import json as _json

for seed in range(12):
    rid = f"orientation_strong_seed{seed}"; d = ENC / rid
    if (d / "backbone.pt").exists():
        m = _json.loads((d / "metrics.json").read_text()); diag = m.get("diagnostics", {})
        print(f"{rid}: DONE loss={m['final_loss']:.4f} nan={m['nan_aborted']} epochs={m['epochs_run']} | "
              f"feat_std={diag.get('feat_std'):.4f} eff_rank={diag.get('eff_rank'):.1f} "
              f"align={diag.get('alignment'):.3f} unif={diag.get('uniformity'):.3f}")
    elif (d / "last_ckpt.pt").exists():
        last = None
        for line in (d / "train_log.jsonl").read_text().splitlines():
            try: last = _json.loads(line).get("epoch", last)
            except Exception: pass
        print(f"{rid}: partial @ep{last}")
    else:
        print(f"{rid}: todo")

## 10. Probe the cell → `stacks.npz` (the interface contract)

In [ ]:
import subprocess, time, glob
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
TAG = "orientation_strong"
N_SEEDS = 12
RANDOM_SEEDS = [str(i) for i in range(N_SEEDS)]

if (REPO / "results" / "probes" / TAG / "stacks.npz").exists():
    print(f"{TAG}: stacks.npz already present -> nothing to do.")
else:
    encs = sorted(glob.glob(str(REPO / f"results/encoders/{TAG}_seed*/backbone.pt")))
    assert len(encs) == N_SEEDS, f"only {len(encs)}/{N_SEEDS} encoders trained -- finish section 7 first"
    cmd = ["python", "-m", "src.probes.run_sweep",
           "--config", "configs/probe/ladder.yaml",
           "--dataset", "shapes3d",
           "--condition", "orientation", "--strength", "strong",
           "--encoders", *encs,
           "--random-seed", *RANDOM_SEEDS,
           "--epochs", "100", "--num-workers", "2"]
    t = time.time()
    rc = subprocess.run(cmd, cwd=str(REPO)).returncode
    print(f"{TAG}: {'OK' if rc == 0 else 'FAILED (exit %s)' % rc} in {(time.time()-t)/3600:.2f} h")

## 11. Save the finished cell